In [ ]:
# =============================================================================
# explore/Explore_critical_sources.py
# Which sources can this workspace actually reach for the critical sites (job 31)? Read only. Writes nothing. Safe to run any time.
# (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). AddressBase Premium is not available to us yet, so critical sites come from:
#   schools and colleges    DfE Get Information About Schools (GIAS)
#   hospitals, GP surgeries NHS Organisation Data Service (ODS) API, with postcodes turned into coordinates by the ONS Postcode Directory
#   care homes              the CQC directory (retry), and OpenStreetMap
#   emergency services, prisons, substations, water and sewage works   OpenStreetMap UK (a .osm.pbf in the lake)
# It tests each source and says what came back. Photograph or copy the result tables back. Nothing here is secret.
# Widgets: osm_file (the OpenStreetMap file).
# =============================================================================

import os
import sys
import time

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import datetime as dt
import pandas as pd
import requests

import config
from utils.helpers import get_spark

spark = get_spark()
HEADERS = {"User-Agent": "FGS-critical-sites-survey"}


def show(df):
    display(df.fillna("").astype(str) if len(df) else df)


dbutils.widgets.text("osm_file", "/Volumes/prd_dash_bronze/openstreetmap_unrestricted/united_kingdom/format_PBF_united_kingdom/LATEST_united_kingdom/", "OpenStreetMap folder or file")
OSM = dbutils.widgets.get("osm_file")


In [ ]:
# =============================================================================
# 1. SCHOOLS: can the GIAS data file be downloaded? The file is dated, so recent days are tried.
# =============================================================================

rows = []
base = "https://ea-edubase-api-prod.azurewebsites.net/edubase/downloads/public/edubasealldata{d}.csv"
for back in range(0, 8):
    d = (dt.date.today() - dt.timedelta(days=back)).strftime("%Y%m%d")
    url = base.format(d=d)
    try:
        r = requests.get(url, timeout=40, stream=True, headers=HEADERS)
        head = next(r.iter_content(2048), b"")
        rows.append({"url": url, "status": r.status_code, "content_type": r.headers.get("content-type", ""),
                     "bytes_first": len(head), "starts": head[:80].decode("latin-1", "replace").replace("\n", " ")})
        if r.status_code == 200:
            break
    except Exception as e:
        rows.append({"url": url, "status": "ERROR", "starts": str(e)[:100]})
print("1. GIAS direct download (the first 200 stops the search):")
show(pd.DataFrame(rows))
try:
    page = requests.get("https://get-information-schools.service.gov.uk/Downloads", timeout=40, headers=HEADERS)
    print("GIAS downloads page:", page.status_code, len(page.text), "bytes")
except Exception as e:
    print("GIAS downloads page:", str(e)[:120])


In [ ]:
# =============================================================================
# 2. NHS ODS API: hospitals and GP practices. Roles: RO198 NHS trust site, RO177 GP practice (prescribing cost centre), RO76 GP practice.
# The API gives name, postcode and status but no coordinates; the ONS Postcode Directory (in the lake) turns postcodes into coordinates.
# =============================================================================

rows = []
for label, role in (("NHS trust site", "RO198"), ("GP practice", "RO76"), ("Prescribing cost centre", "RO177")):
    url = f"https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations?PrimaryRoleId={role}&Status=Active&Limit=5&Offset=0"
    try:
        r = requests.get(url, timeout=40, headers=HEADERS)
        j = r.json() if r.status_code == 200 else {}
        orgs = j.get("Organisations", [])
        rows.append({"role": label, "code": role, "status": r.status_code, "returned": len(orgs),
                     "total_header": r.headers.get("X-Total-Count", ""), "first": (orgs[0] if orgs else {}).get("Name", ""),
                     "keys": ", ".join(sorted(orgs[0].keys())) if orgs else ""})
    except Exception as e:
        rows.append({"role": label, "code": role, "status": "ERROR", "first": str(e)[:100]})
print("2. NHS ODS API:")
show(pd.DataFrame(rows))

# how a postcode becomes coordinates: the ONSPD in the lake
ONS = "/Volumes/prd_dash_bronze/ons_open_geography_portal_unrestricted/ons_postcode_directory/format_PARQUET_ons_postcode_directory/LATEST_ons_postcode_directory/"
try:
    pc = spark.read.parquet(ONS).select("pcds", "east1m", "north1m", "doterm", "ctry26cd").limit(3).toPandas()
    print("ONS Postcode Directory columns for coordinates:")
    show(pc)
except Exception as e:
    print("ONSPD read failed:", str(e)[:150])


In [ ]:
# =============================================================================
# 3. CARE HOMES: CQC. The public API returned a 502 before: try again, and look for the data download page.
# =============================================================================

rows = []
for name, url in (("CQC API providers", "https://api.service.cqc.org.uk/public/v1/providers?perPage=1"),
                  ("CQC API locations", "https://api.service.cqc.org.uk/public/v1/locations?perPage=1"),
                  ("CQC data page", "https://www.cqc.org.uk/about-us/transparency/using-cqc-data")):
    for attempt in range(2):
        try:
            r = requests.get(url, timeout=40, headers=HEADERS)
            rows.append({"source": name, "attempt": attempt + 1, "status": r.status_code, "bytes": len(r.content), "starts": r.text[:100].replace("\n", " ")})
            if r.status_code == 200:
                break
        except Exception as e:
            rows.append({"source": name, "attempt": attempt + 1, "status": "ERROR", "starts": str(e)[:100]})
        time.sleep(3)
print("3. CQC:")
show(pd.DataFrame(rows))


In [ ]:
# =============================================================================
# 4. OPENSTREETMAP: is the file there, how big, and can it be read here?
# Emergency services, prisons, substations, water and sewage works, and care homes and clinics, come from this file.
# =============================================================================

def ls(path):
    try:
        return [(f.path.replace("dbfs:", ""), round(f.size / 1e6, 1)) for f in dbutils.fs.ls(path)]
    except Exception as e:
        return [(path, "ERROR " + str(e)[:100])]


print("4. OpenStreetMap file(s):")
show(pd.DataFrame(ls(OSM), columns=["path", "size_mb"]))

tools = []
for mod, pipname in (("osmium", "osmium"), ("pyrosm", "pyrosm"), ("geopandas", "geopandas"), ("pyogrio", "pyogrio"), ("fiona", "fiona")):
    try:
        m = __import__(mod)
        tools.append({"module": mod, "available": True, "version": getattr(m, "__version__", "")})
    except Exception as e:
        tools.append({"module": mod, "available": False, "version": "not installed (pip name: " + pipname + ")"})
print("Python modules that can read a .osm.pbf:")
show(pd.DataFrame(tools))
try:
    import pyogrio
    drivers = pyogrio.list_drivers()
    print("GDAL has an OSM driver:", "OSM" in drivers)
except Exception:
    print("pyogrio not available to check for the GDAL OSM driver. A pip install of osmium (pyosmium) is the likely route.")
print("\nDone. Send the tables back. Nothing was written.")
dbutils.notebook.exit("success")
